In [1]:
import numpy as np
import pandas as pd
from Korpora import Korpora


corpus = Korpora.load("nsmc")
df = pd.DataFrame(corpus.test).sample(20000, random_state=42)


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/

[Korpora] Corpus `nsmc` is already installed at /home/teom142/Korpora/nsmc/ratings_train.txt
[Korpora] Corpus `nsmc` is already installed at /home/teom142

In [2]:
train, valid, test = np.split(
    df.sample(frac=1, random_state=42), [int(0.6 * len(df)), int(0.8 * len(df))]
)

print(train.head(5).to_markdown())
print(f"Training Data Size : {len(train)}")
print(f"Validation Data Size : {len(valid)}")
print(f"Testing Data Size : {len(test)}")

|       | text                                                     |   label |
|------:|:---------------------------------------------------------|--------:|
| 26891 | 역시 코믹액션은 성룡, 홍금보, 원표 삼인방이 최고지!!     |       1 |
| 25024 | 점수 후하게 줘야것네 별 반개~                            |       0 |
| 11666 | 오랜만에 느낄수 있는 [감독] 구타욕구.                    |       0 |
| 40303 | 본지는 좀 됬지만 극장서 돈주고 본게 아직까지 아까운 영화 |       0 |
| 18010 | 징키스칸이란 소재를 가지고 이것밖에 못만드냐             |       0 |
Training Data Size : 12000
Validation Data Size : 4000
Testing Data Size : 4000


In [ ]:
import torch  # PyTorch 라이브러리
from transformers import BertTokenizer  # BERT 모델 토크나이저 불러오기
from torch.utils.data import TensorDataset, DataLoader  # 텐서 묶기 및 배치 단위 불러오기 도구
from torch.utils.data import RandomSampler, SequentialSampler  # 데이터 샘플링 도구 (무작위, 순차)

# make_dataset 함수: 데이터(data)를 토크나이징하고 TensorDataset으로 변환하는 함수
def make_dataset(data, tokenizer, device):
    # 데이터의 text 열을 리스트로 변환하여 토크나이저에 입력
    # padding="longest": 배치 내 가장 긴 문장 길이에 맞춰 패딩 추가
    # truncation=True: 너무 긴 문장 자르기
    # return_tensors="pt": 결과를 PyTorch 텐서 형태로 반환
    tokenized = tokenizer(
        text=data.text.tolist(),
        padding="longest",
        truncation=True,
        return_tensors="pt"
    )
    
    # 토큰화 결과에서 input_ids 추출 후 device 이동
    input_ids = tokenized["input_ids"].to(device)
    # 토큰화 결과에서 attention_mask 추출 후 device 이동
    attention_mask = tokenized["attention_mask"].to(device)
    # 데이터의 label 열을 정수형 텐서로 변환 후 device 이동
    labels = torch.tensor(data.label.values, dtype=torch.long).to(device)
    
    # input_ids, attention_mask, labels를 하나의 TensorDataset으로 묶음
    return TensorDataset(input_ids, attention_mask, labels)

# get_datalodader 함수: 주어진 데이터셋과 샘플러로 DataLoader 생성하는 함수
def get_datalodader(dataset, sampler, batch_size):
    # 지정한 sampler(RandomSampler 또는 SequentialSampler)를 데이터셋에 적용
    data_sampler = sampler(dataset)
    # DataLoader 생성: dataset을 sampler 및 batch_size에 따라 배치 단위로 불러옴
    dataloader = DataLoader(dataset, sampler=data_sampler, batch_size=batch_size)
    return dataloader

# 학습 관련 하이퍼파라미터 설정
epochs = 5          # 전체 학습 에폭 수 (데이터셋 반복 횟수)
batch_size = 32     # 한 배치에 포함되는 데이터 샘플 수

# 사용할 디바이스 설정: CUDA(GPU) 사용 가능 시 "cuda", 그렇지 않으면 "cpu" 선택
device = "cuda" if torch.cuda.is_available() else "cpu"

# 사전 학습된 BERT 모델 중 다국어 모델(bert-base-multilingual-cased) 토크나이저 불러오기
# do_lower_case=False: 대소문자 구분
tokenizer = BertTokenizer.from_pretrained(
    pretrained_model_name_or_path="bert-base-multilingual-cased",
    do_lower_case=False
)

# 각 변수는 학습(train), 검증(valid), 테스트(test) 데이터셋 (예: DataFrame 형태의 데이터)을 의미

# 학습 데이터셋 생성: make_dataset 함수로 텍스트와 레이블 토크나이징 및 TensorDataset 변환
train_dataset = make_dataset(train, tokenizer, device)
# RandomSampler 사용: 학습 데이터셋에서 무작위로 데이터 선택하는 DataLoader 생성
train_dataloader = get_datalodader(train_dataset, RandomSampler, batch_size)

# 검증 데이터셋 생성
valid_dataset = make_dataset(valid, tokenizer, device)
# SequentialSampler 사용: 검증 데이터셋의 데이터를 순차적으로 불러오는 DataLoader 생성
valid_dataloader = get_datalodader(valid_dataset, SequentialSampler, batch_size)

# 테스트 데이터셋 생성
test_dataset = make_dataset(test, tokenizer, device)
# SequentialSampler 사용: 테스트 데이터셋의 데이터를 순차적으로 불러오는 DataLoader 생성
test_dataloader = get_datalodader(test_dataset, SequentialSampler, batch_size)

# 학습 데이터셋의 첫 번째 샘플(토큰화된 텍스트, attention mask, 레이블) 출력하여 데이터 구조 확인
print(train_dataset[0])


/home/teom142/.conda/envs/pytorch_study/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/teom142/.conda/envs/pytorch_study/lib/python3.10/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


(tensor([   101,  58466,   9812, 118956, 119122,  59095,  10892,   9434, 118888,
           117,   9992,  40032,  30005,    117,   9612,  37824,   9410,  12030,
         42337,  10739,  83491,  12508,    106,    106,    102,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,

In [4]:
from torch import optim
from transformers import BertForSequenceClassification


model = BertForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path="bert-base-multilingual-cased",
    num_labels=2
).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-5, eps=1e-8)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
for main_name, main_module in model.named_children():
    print(main_name)
    for sub_name, sub_module in main_module.named_children():
        print("└", sub_name)
        for ssub_name, ssub_module in sub_module.named_children():
            print("│  └", ssub_name)
            for sssub_name, sssub_module in ssub_module.named_children():
                print("│  │  └", sssub_name)

bert
└ embeddings
│  └ word_embeddings
│  └ position_embeddings
│  └ token_type_embeddings
│  └ LayerNorm
│  └ dropout
└ encoder
│  └ layer
│  │  └ 0
│  │  └ 1
│  │  └ 2
│  │  └ 3
│  │  └ 4
│  │  └ 5
│  │  └ 6
│  │  └ 7
│  │  └ 8
│  │  └ 9
│  │  └ 10
│  │  └ 11
└ pooler
│  └ dense
│  └ activation
dropout
classifier


In [ ]:
import numpy as np  # NumPy 라이브러리
from torch import nn  # torch.nn 모듈 (신경망 관련 기능)

def calc_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()  # 예측값의 인덱스 추출 후 평탄화
    labels_flat = labels.flatten()  # 실제 레이블 평탄화
    return np.sum(pred_flat == labels_flat) / len(labels_flat)  # 정확도 계산

def train(model, optimizer, dataloader):
    model.train()  # 모델 학습 모드 설정
    train_loss = 0.0  # 학습 손실 초기화

    for input_ids, attention_mask, labels in dataloader:
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)  # 모델에 입력값 전달 및 출력 획득

        loss = outputs.loss  # 출력에서 손실 값 추출
        train_loss += loss.item()  # 손실 값 누적
        
        optimizer.zero_grad()  # 옵티마이저 기울기 초기화
        loss.backward()  # 역전파를 통한 기울기 계산
        optimizer.step()  # 가중치 업데이트

    train_loss = train_loss / len(dataloader)  # 에폭당 평균 손실 계산
    return train_loss  # 평균 학습 손실 반환

def evaluation(model, dataloader):
    with torch.no_grad():  # 평가 시 그래디언트 계산 비활성화
        model.eval()  # 모델 평가 모드 설정
        criterion = nn.CrossEntropyLoss()  # 교차 엔트로피 손실 함수 정의
        val_loss, val_accuracy = 0.0, 0.0  # 검증 손실 및 정확도 초기화
        
        for input_ids, attention_mask, labels in dataloader:
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)  # 모델에 입력값 전달 및 출력 획득
            logits = outputs.logits  # 모델 출력 로짓 추출

            loss = criterion(logits, labels)  # 로짓과 실제 레이블 간 손실 계산
            logits = logits.detach().cpu().numpy()  # 로짓 분리 후 CPU 이동 및 NumPy 배열 변환
            label_ids = labels.to("cpu").numpy()  # 레이블 CPU 이동 및 NumPy 배열 변환
            accuracy = calc_accuracy(logits, label_ids)  # 정확도 계산
            
            val_loss += loss.item()  # 손실 값 누적
            val_accuracy += accuracy  # 정확도 값 누적
    
    val_loss = val_loss / len(dataloader)  # 평균 검증 손실 계산
    val_accuracy = val_accuracy / len(dataloader)  # 평균 검증 정확도 계산
    return val_loss, val_accuracy  # 평균 검증 손실 및 정확도 반환

best_loss = 10000  # 초기 최적 손실 값 설정
for epoch in range(epochs):
    train_loss = train(model, optimizer, train_dataloader)  # 학습 함수 호출 및 평균 학습 손실 계산
    val_loss, val_accuracy = evaluation(model, valid_dataloader)  # 평가 함수 호출 및 평균 검증 손실, 정확도 계산
    print(f"Epoch {epoch + 1}: Train Loss: {train_loss:.4f} Val Loss: {val_loss:.4f} Val Accuracy {val_accuracy:.4f}")  # 학습 및 평가 결과 출력

    if val_loss < best_loss:  # 현재 검증 손실이 최적 손실보다 낮을 경우
        best_loss = val_loss  # 최적 손실 값 업데이트
        torch.save(model.state_dict(), "../models/BertForSequenceClassification.pt")  # 모델 가중치 저장
        print("Saved the model weights")  # 모델 저장 결과 출력


Epoch 1: Train Loss: 0.5437 Val Loss: 0.4513 Val Accuracy 0.7887
Saved the model weights
Epoch 2: Train Loss: 0.4091 Val Loss: 0.4148 Val Accuracy 0.8093
Saved the model weights
Epoch 3: Train Loss: 0.3220 Val Loss: 0.4168 Val Accuracy 0.8167
Epoch 4: Train Loss: 0.2545 Val Loss: 0.4749 Val Accuracy 0.8163
Epoch 5: Train Loss: 0.1969 Val Loss: 0.5138 Val Accuracy 0.8223


In [7]:
model = BertForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path="bert-base-multilingual-cased",
    num_labels=2
).to(device)
model.load_state_dict(torch.load("../models/BertForSequenceClassification.pt"))

test_loss, test_accuracy = evaluation(model, test_dataloader)
print(f"Test Loss : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy:.4f}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Test Loss : 0.4054
Test Accuracy : 0.8095
